In [31]:
import pandas as pd

In [32]:
src = "../../textos/tedlock-text-only.txt"

In [33]:
lines = [line.strip() for line in open(src,'r').readlines()]

In [34]:
LINE = pd.DataFrame(lines, columns=['line_str'])
LINE.loc[LINE.line_str.str.match("^\s*PART"), 'part'] = LINE.line_str
LINE.part = LINE.part.ffill()
LINE = LINE[~LINE.line_str.str.match("^\s*PART")]

In [42]:
replacers = {f"PART {num}":i+1 for i, num in enumerate("ONE TWO THREE FOUR FIVE".split())}

In [43]:
replacers

{'PART ONE': 1, 'PART TWO': 2, 'PART THREE': 3, 'PART FOUR': 4, 'PART FIVE': 5}

In [45]:
LINE.part = LINE.part.map(replacers)

In [46]:
LINE

,line_str,part
1,"THIS IS THE BEGINNING OF THE ANCIENT WORD, her...",1
2,"called Quiche. Here we shall inscribe, we shal...",1
3,"Ancient Word, the potential and source for eve...",1
4,"citadel of Quiche, in the nation of Quiche peo...",1
5,"And here we shall take up the demonstration, r...",1
...,...,...
3897,"THIS IS ENOUGH ABOUT THE BEING OF QUICHE, give...",5
3898,longer a place to see it. There is the origina...,5
3899,"writing owned by the lords, now lost, but even...",5
3900,"has been completed here concerning Quiche, whi...",5


In [53]:
PART = LINE.groupby(['part']).line_str.apply(lambda x: ' '.join(map(str,x))).to_frame('part_str')

In [54]:
PART

,part_str
part,
1,"THIS IS THE BEGINNING OF THE ANCIENT WORD, her..."
2,HERE IS THE BEGINNING OF THE DEFEAT AND DESTRU...
3,AND NOW WE SHALL NAME THE NAME OF THE FATHER O...
4,AND HERE IS THE BEGINNING OF THE CONCEPTION OF...
5,AND THEN THEY REMEMBERED WHAT HAD BEEN SAID AB...


In [86]:
import spacy
nlp = spacy.load("en_core_web_lg")

In [ ]:
SENT = PART.part_str.apply(nlp).apply(lambda x: [sent for sent in x.sents])\
    .apply(pd.Series).stack().to_frame('sent_str')
SENT.index.names = PART.index.names[:1] + ['sent']
TOKEN = SENT.sent_str.apply(lambda x: [[token.text, token.pos_, token.dep_] for token in x])\
    .apply(pd.Series).stack()\
    .apply(pd.Series)
TOKEN.columns = ['token_str','token_pos','token_dep']
TOKEN.index.names = TOKEN.index.names[:2] + ['token_num']

In [92]:
TOKEN

token_str token_pos token_dep
part sent token_num                               
1    0    0               THIS      PRON     nsubj
          1                 IS       AUX      ROOT
          2                THE       DET       det
          3          BEGINNING      NOUN      attr
          4                 OF       ADP      prep
...                        ...       ...       ...
5    419  30               now       ADV    advmod
          31             named      VERB     relcl
          32             Santa     PROPN  compound
          33              Cruz     PROPN      oprd
          34                 .     PUNCT     punct

[48076 rows x 3 columns]

In [95]:
DOC = TOKEN.groupby(['part','sent']).token_str.apply(lambda x: ' '.join(map(str,x)))\
    .to_frame('sent_str')

In [96]:
DOC

sent_str
part sent                                                   
1    0     THIS IS THE BEGINNING OF THE ANCIENT WORD , he...
     1     Here we shall inscribe , we shall implant the ...
     2     And here we shall take up the demonstration , ...
     3     They accounted for everythingand did it , too-...
     4     We shall write about this now amid the preachi...
...                                                      ...
5    415   And Great Toastmaster for the Greathouses , se...
     416   Great Toastmaster Lord for the Lord Quiches , ...
     417   And so there are three Toastmasters , one repr...
     418   THIS IS ENOUGH ABOUT THE BEING OF QUICHE , giv...
     419   There is the original book and ancient writing...

[2792 rows x 1 columns]

In [100]:
DOC = SENT.sent_str.apply(lambda x: x.text).to_frame('doc_str')

In [102]:
DOC.to_csv("tedlock-LINE.csv", index=True)